In [1]:
# Load data from the spreadsheet
import sys
import pandas as pd
from pathlib import Path
import openpyxl

from loguru import logger
logger.remove()
logger.add(sys.stderr, level="ERROR")

filename = Path(r"C:\Users\crdig\Power Concepts NZ Ltd\Files - Product Management\Old product data\BOMs\Master files\20240916 POWER CONCEPTS BOM COSTINGS - V02.xlsx")
assert filename.exists(), f"File {filename} does not exist."

# Import the data
df_raw = pd.read_excel(filename, sheet_name="Sheet1", skiprows=0, usecols="A:EL", header=None, dtype=object)
# df_raw

In [2]:
# Go through the header rows and ffil them to support merged cells
df_filled = df_raw.copy()
df_filled.iloc[0] = df_filled.iloc[0].ffill()
df_filled.iloc[1] = df_filled.iloc[1].ffill()
df_filled.iloc[2] = df_filled.iloc[2].ffill()

# Use the first 4 rows as a MultiIndex for the columns
df_filled.columns = pd.MultiIndex.from_arrays(df_filled.iloc[:4].values)

# Drop the first 4 rows since they are now headers
df_filled = df_filled.iloc[4:]

df_filled


NaN                            \
            NaN                             
            NaN Part Information            
    item number      Part Number Revision   
4             1           200874        3   
5             2           200875        6   
6             3           200876        9   
7             4           200877        1   
8             5           200878        2   
..          ...              ...      ...   
204         201        91828A415      NaN   
205         202        90225A102      NaN   
206         203        97654A382      NaN   
207         204        96315A116      NaN   
208         205        97654A381      NaN   

                                                                              \
                                                                               
                                                                               
                               Description                          Material   
4             Extrusion,Battery Module Top  Al6063-T5 or approved equivelent   
5          Extrusion,Battery Module Bottom  Al6063-T5 or approved equivelent   
6           Heat Sink, Battery Module Rear     AL6061 or approved equivelent   
7    Mount, Fuse Flexible – Battery Module     AL6061 or approved equivelent   
8       Mount, Fuse Fixed – Battery Module     AL6061 or approved equivelent   
..                                     ...                               ...   
204                               Nut, M10                  Stainless Steel    
205                      Nut, Square - M6                   Stainless Steel    
206             Screw, Flange Head M6 X 20                  Stainless Steel    
207                           Nut, Thin M6                  Stainless Steel    
208             Screw, Flange Head M6 X 16                  Stainless Steel    

                                                                  ...  \
                                                                  ...   
             Quote Quantites                    Module Quantites  ...   
      Module             MOQ 500 Unit 1000 Unit                B  ...   
4    Battery             250     2000      4000                1  ...   
5    Battery             250     2000      4000                1  ...   
6    Battery             250     2000      4000                1  ...   
7    Battery             250     2000      4000                1  ...   
8    Battery             250     2000      4000                1  ...   
..       ...             ...      ...       ...              ...  ...   
204     Rack             NaN      NaN       NaN                0  ...   
205     Rack             NaN      NaN       NaN                0  ...   
206     Rack             NaN      NaN       NaN                0  ...   
207     Rack             NaN      NaN       NaN                0  ...   
208    Rack              NaN      NaN       NaN                0  ...   

                 ANGLO                           AXIEM                         \
                   USD                             USD                          
    LASER CUT AND FOLD              LASER CUT AND FOLD                          
                  1000 Tooling Cost                MOQ  500 1000 Tooling Cost   
4                  NaN          NaN                NaN  NaN  NaN          NaN   
5                  NaN          NaN                NaN  NaN  NaN          NaN   
6                  NaN          NaN                NaN  NaN  NaN          NaN   
7                  NaN          NaN                NaN  NaN  NaN          NaN   
8                  NaN          NaN                NaN  NaN  NaN          NaN   
..                 ...          ...                ...  ...  ...          ...   
204                NaN          NaN                NaN  NaN  NaN          NaN   
205                NaN          NaN                NaN  NaN  NaN          NaN   
206                NaN          NaN                NaN  NaN  NaN   

In [3]:
# Overview the column structure (If desired)
nan_columns = df_filled.columns[df_filled.columns.to_frame().isna().any(axis=1)]

# nan_columns

In [ ]:
# Add parts to PartsBox and then to assemblies
import sys
from PartsBoxAPI.PartsBoxAPI import PartsBoxAPI
from requests.exceptions import HTTPError
# Add all non-existing parts into partsbox
tags = ["Mechanical"]  # You need to create the tag in PartsBox first

logger.remove()
logger.add(sys.stderr, level="ERROR")

# Check assemblies are valid and expected
PartsBox = PartsBoxAPI("partsboxapi_6cf6evkbnmhr4a70pqxzd2hcvab3bda5c95114707b905ab70858635cb5cd549d")  # Real API key
# PartsBox = PartsBoxAPI("partsboxapi_8dvrkmg1tekt3973dpga6c2exee90d5a4d82ed8964decb742a7ff518c85d9326")  # Test API key

Assembly_IDs = {
    "Battery":["cdzs5qj2jaj35a12brdr76s0yc"],
    "Charger":["dyxbhq3py8ha1bdmthv0w4jre9"],
    "Inverter":["b3k8xkjk7eggz8g7p0hwsket2v"],
    # "Rack 1x6": Not existent yet
    "Rack 2x3":["1wgxww7m4tkwvbd842w6ysd68b"],
}

for key, value in Assembly_IDs.items():
    value = value[0]
    try:
        response = pd.DataFrame(PartsBox.projects.get_project_entries(value)['data'])
        print(f"Assembly {key} with ID {value} found in PartsBox.")
        Assembly_IDs[key].append(response)
    except HTTPError as e:
        print(f"NOT FOUND - Assembly {key} with ID {value} {e}")

database = pd.DataFrame(PartsBox.parts.get_all_parts()['data'])

for index, row in df_filled.iterrows():
    # Skip the first 4 rows, as they are headers

    # Get the part number and description
    part_number = str(row.iloc[1])
    description = row.iloc[3]
    
    assembly_data = {
        "Battery": row.iloc[9],
        "Charger": row.iloc[10],
        "Inverter": row.iloc[11],
        # "Rack 1x6": row[12],
        "Rack 2x3": row.iloc[13],
    }
    print(part_number, assembly_data)
    
    # Check if the part number already exists in PartsBox
    result = database.loc[database['part/name'] == part_number]
    if len(result) == 0: # Item not found in database, so create
        item_id = PartsBox.parts.create_part(
            part_type="local",
            part_name=str(part_number),
            part_description=description,
            part_tags=tags,
            )['data']['part/id']
    else:
        item_id = result.iloc[0]['part/id']
    
    # Now update the quantities in assemblies
    for assembly, quantity in assembly_data.items():
        assembly_key = Assembly_IDs[assembly][0]
        assembly_data = Assembly_IDs[assembly][1]
        
        try:
            matching_entry = assembly_data.loc[assembly_data["entry/part-id"] == item_id]
        except KeyError:
            matching_entry = []
        
        if len(matching_entry) == 0:  # This is a new part in the assembly
            if quantity > 0:
                response = PartsBox.projects.add_project_entries(
                    project_id=assembly_key,
                    entries = [
                        {
                            "entry/part-id": item_id,
                            "entry/quantity": int(quantity),
                            "entry/designators": [part_number+"_"+str(i) for i in range(int(quantity))],
                        }
                    ]
                )  
        else: # This is an existing part in an assembly
            if quantity == 0: # Remove the entry if quantity is 0
                PartsBox.projects.delete_project_entries(
                    project_id=assembly_key,
                    ids=list(matching_entry["entry/id"]),
                )
            else: # Update the quantity
                PartsBox.projects.update_project_entries(
                    project_id=assembly_key,
                    entries=[
                        {
                            "entry/id": matching_entry["entry/id"].iloc[0],
                            "entry/part-id": item_id,
                            "entry/quantity": int(quantity),
                            "entry/designators": [part_number+"_"+str(i) for i in range(int(quantity))],
                        }
                    ]
                )
    pass




Assembly Battery with ID cdzs5qj2jaj35a12brdr76s0yc found in PartsBox.
Assembly Charger with ID dyxbhq3py8ha1bdmthv0w4jre9 found in PartsBox.
Assembly Inverter with ID b3k8xkjk7eggz8g7p0hwsket2v found in PartsBox.
Assembly Rack 2x3 with ID 1wgxww7m4tkwvbd842w6ysd68b found in PartsBox.
200874 {'Battery': 1, 'Charger': 0, 'Inverter': 0, 'Rack 2x3': 0}
200875 {'Battery': 1, 'Charger': 0, 'Inverter': 0, 'Rack 2x3': 0}
200876 {'Battery': 1, 'Charger': 0, 'Inverter': 0, 'Rack 2x3': 0}
200877 {'Battery': 1, 'Charger': 0, 'Inverter': 0, 'Rack 2x3': 0}
200878 {'Battery': 1, 'Charger': 0, 'Inverter': 0, 'Rack 2x3': 0}
200885 {'Battery': 2, 'Charger': 2, 'Inverter': 2, 'Rack 2x3': 0}
202475 {'Battery': 2, 'Charger': 2, 'Inverter': 2, 'Rack 2x3': 0}
200766 {'Battery': 2, 'Charger': 0, 'Inverter': 0, 'Rack 2x3': 0}
200767 {'Battery': 1, 'Charger': 0, 'Inverter': 0, 'Rack 2x3': 0}
200794 {'Battery': 1, 'Charger': 0, 'Inverter': 0, 'Rack 2x3': 0}
200795 {'Battery': 2, 'Charger': 4, 'Inverter': 4, 'Ra

In [ ]:
# Populate pricing data
